# Pipeline de Dados — MLP do Zero (AI4Good)

Este notebook documenta todo o pipeline: **extração**, **pré-processamento** e **treinamento/validação base** da rede MLP construída manualmente em numpy.

As bases usadas:
- `heart.csv` (Heart Disease) — base principal (1ª etapa)
- `diabetic_data.csv` + `IDS_mapping.csv` (Diabetes 130-US Hospitals) — desafio (2ª etapa)

Split obrigatório: **80% treino / 20% teste**.

In [ ]:
import sys
from pathlib import Path

PROJETO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJETO) not in sys.path:
    sys.path.insert(0, str(PROJETO))

import numpy as np
import pandas as pd
from src.data.load_data import load_heart, load_diabetic, load_ids_mapping
from src.data.preprocess import (
    preprocess_heart, PreprocessDiabetic, split_train_test,
    StandardScaler, clip_outliers,
)
from src.model.mlp import MLP
from src.model.trainer import Trainer
from src.config import RANDOM_STATE

print('Ambiente pronto.')

Ambiente pronto.


## 1. Extração de Dados

Carregamento das bases brutas.

In [ ]:
# 1.1 Base principal: Heart Disease
heart = load_heart()
print('heart.csv:', heart.shape)
print('Missings:', int(heart.isna().sum().sum()))
print('Distribuição do target:')
print(heart['target'].value_counts())

heart.csv: (1025, 14)


Missings: 0
Distribuição do target:
target
1    526
0    499
Name: count, dtype: int64


In [ ]:
# 1.2 Desafio: Diabetes + dicionário de IDs
diabetic = load_diabetic()
ids = load_ids_mapping()
print('diabetic_data.csv:', diabetic.shape)
print('Células com \'?\':', int((diabetic == '?').sum().sum()))
print('Seções do IDS_mapping:', {k: len(v) for k, v in ids.items()})
print('readmitted:')
print(diabetic['readmitted'].value_counts())

diabetic_data.csv: (101766, 50)
Células com '?': 192849
Seções do IDS_mapping: {'admission_type_id': 8, 'discharge_disposition_id': 30, 'admission_source_id': 25}
readmitted:
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64


## 2. Pré-processamento

### 2.1 Heart Disease
A base `heart.csv` já é numérica e sem missings. O tratamento aplicado: winsorização de outliers (IQR) e padronização (z-score) ajustada apenas no treino.

In [ ]:
X_h, y_h = preprocess_heart(heart)
X_h = clip_outliers(X_h, ['age', 'trestbps', 'chol', 'thalach', 'oldpeak'])

X_tr, X_te, y_tr, y_te = split_train_test(X_h, y_h, random_state=RANDOM_STATE)
print('Split -> treino:', X_tr.shape, '| teste:', X_te.shape)

scaler = StandardScaler().fit(X_tr.values)
X_tr_n = scaler.transform(X_tr.values)
X_te_n = scaler.transform(X_te.values)
print('Médias após padronização (treino):', np.round(X_tr_n.mean(axis=0), 3)[:6])
print('Desvios após padronização (treino):', np.round(X_tr_n.std(axis=0), 3)[:6])

Split -> treino: (820, 13) | teste: (205, 13)
Médias após padronização (treino): [ 0.  0. -0. -0. -0.  0.]
Desvios após padronização (treino): [1. 1. 1. 1. 1. 1.]


## 3. Treinamento e Validação Base

MLP construída do zero (`src/model/mlp.py`): inicialização He/Xavier, feed-forward, back-propagation e gradient descent manuais. Implementação validada por gradiente numérico (diferenças finitas).

In [ ]:
modelo = MLP([13, 8, 4, 1], hidden_activation='relu', output_activation='sigmoid', seed=RANDOM_STATE)
trainer = Trainer(modelo, lr=0.05, batch_size=None, seed=RANDOM_STATE)
historico = trainer.fit(X_tr_n, y_tr.to_numpy(), X_te_n, y_te.to_numpy(), epochs=150, lr=0.05)

df_hist = pd.DataFrame(historico)
print(df_hist.tail(3).round(4).to_string(index=False))

  loss  accuracy  val_loss  val_accuracy  epoch
0.5679    0.7488    0.5594         0.761    148
0.5665    0.7439    0.5578         0.761    149
0.5651    0.7439    0.5562         0.761    150


In [ ]:
# Avaliação final do melhor modelo (pesos restaurados pelo Trainer)
from src.viz.graph_plot import plot_network, plot_history
import matplotlib.pyplot as plt

acc_test = trainer._accuracy(X_te_n, y_te.to_numpy())
print(f'Acurácia no teste (20%): {acc_test:.4f}')

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
plot_history(historico, axs=axs)
plt.tight_layout()
plt.show()

Acurácia no teste (20%): 0.7610


/tmp/ipykernel_2147/638043250.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Desafio: Diabetes 130-US Hospitals

Pré-processamento com pandas: remoção de identificadores/colunas com alto missing, mapeamento dos IDs via `IDS_mapping.csv`, agrupamento dos diagnósticos ICD-9, ordinalização de medicações e padronização.

In [ ]:
preproc = PreprocessDiabetic(binary_target=True)
X_d, y_d = preproc.transform(diabetic, ids)
print('Features do diabetes:', X_d.shape, '| n_features =', len(preproc.feature_names_))
print('Target (<30 días = 1):')
print(y_d.value_counts())

Features do diabetes: (101766, 97) | n_features = 97
Target (<30 días = 1):
readmitted
0    90409
1    11357
Name: count, dtype: int64


In [ ]:
X_dt, X_de, y_dt, y_de = split_train_test(X_d, y_d, random_state=RANDOM_STATE)
X_dt = clip_outliers(X_dt, ['time_in_hospital', 'num_lab_procedures', 'num_medications', 'number_diagnoses'])
X_de = clip_outliers(X_de, ['time_in_hospital', 'num_lab_procedures', 'num_medications', 'number_diagnoses'])

scaler_d = StandardScaler().fit(X_dt.values)
X_dt_n = scaler_d.transform(X_dt.values)
X_de_n = scaler_d.transform(X_de.values)
print('Split -> treino:', X_dt_n.shape, '| teste:', X_de_n.shape)

Split -> treino: (81413, 97) | teste: (20353, 97)


In [ ]:
# Treino base rápido no desafio (batch pequeno por desempenho)
modelo_d = MLP([X_dt_n.shape[1], 16, 1], hidden_activation='relu', output_activation='sigmoid', seed=RANDOM_STATE)
trainer_d = Trainer(modelo_d, lr=0.02, batch_size=256, seed=RANDOM_STATE)
historico_d = trainer_d.fit(X_dt_n, y_dt.to_numpy(), X_de_n, y_de.to_numpy(), epochs=10, lr=0.02)
print('Última época:', {k: round(v, 4) for k, v in historico_d[-1].items() if k != 'epoch'})

Última época: {'loss': np.float64(0.3319), 'accuracy': 0.8884, 'val_loss': np.float64(0.3345), 'val_accuracy': 0.8876}


## 5. Conclusões

- O fluxo completo (extração → pré-processamento → treino/validação) está documentado e reaproveitado pela interface Streamlit.
- A MLP do zero foi validada numericamente e alcança acurácia de ~71-75% no teste do Heart Disease com arquitetura enxuta; arquiteturas maiores tendem a melhorar o treino mas prejudicar o teste (overfitting), como previsto na especificação.
- No Diabetes, o alvo binário considera readmissão **<30 dias**; o desbalanceamento (~11% positivos) deve ser considerado ao interpretar a acurácia.